In [1]:
import numpy as np
import pyvista as pv


def plot_data(
        data: np.ndarray | list,
        size: float = 5.0,
):
    coords = data
    colors = np.clip(data, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_clustered_pv(
        data: np.ndarray,
        labels: np.ndarray,
        cluster_centers: np.ndarray,
        size: float = 4.0
):
    coords = data
    cluster_colors = cluster_centers[labels]
    colors = np.clip(cluster_colors, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_centroids_pv(
        cluster_centers: np.ndarray,
        size: float = 18.0
):
    coords = cluster_centers
    colors = np.clip(cluster_centers, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size,
        render_points_as_spheres=True
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()

In [2]:
import numpy as np

def generate_dataset(centers, stds, n_samples=500, ndim=3):
    X = []
    y = []

    for i, center in enumerate(centers):
        cluster = np.random.normal(loc=center, scale=stds[i], size=(n_samples // len(centers), ndim))
        X.append(cluster)
        y.append(np.full(n_samples // len(centers), i))

    return np.vstack(X), np.concatenate(y)

In [3]:
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

default_colors = np.array(plt.colormaps.get_cmap('tab10').colors)

In [4]:
data, ground_truth = generate_dataset(
    [[1, 1, 1], [3, 3, 3], [2, 2, 1]],
    stds=[.5, .5, .5],
    n_samples=500
)

pd.DataFrame(data)

,0,1,2
0,0.461194,-0.229628,2.298712
1,0.251083,1.225753,0.915473
2,1.551630,0.358403,1.509884
3,0.432344,0.510228,0.633646
4,0.806958,1.403570,0.535747
...,...,...,...
493,1.645576,1.912390,-0.353409
494,1.871790,2.231171,0.369071
495,2.332222,1.877665,1.103905
496,2.632063,2.395638,0.706482


In [5]:
plotter = pv.Plotter()

for n, d in zip(ground_truth, data):
    cloud = pv.PolyData(d)
    color = default_colors[n % len(default_colors)]

    plotter.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=10
    )

plotter.add_axes()
plotter.show_grid()
plotter.show()

Widget(value='<iframe src="http://localhost:44965/index.html?ui=P_0x7f01bd3817f0_0&reconnect=auto" class="pyvi…

In [6]:
from ktree.ntree import NTreeDynamic

tree = NTreeDynamic(1)

for a in data:
    tree.insert(a)

sorted_data = tree.sort()
sorted_data = sorted(sorted_data, key=lambda x: len(x))[::-1]
all_clusters = sorted_data

In [7]:
import collections

d_clusters = []

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    centroid = np.mean(s_data, axis=0)

    d_clusters.append((n, centroid))

s_clusters = []

"""

d_clusters = collections.deque(d_clusters)
while len(d_clusters) > 0:
    current_dist = []
    ia, centroid_a = d_clusters.popleft()

    l_clusters = collections.deque(d_clusters)

    while len(l_clusters) > 0:
        ib, centroid_b = l_clusters.popleft()
        dist = np.linalg.norm(centroid_a[1] - centroid_b[1])

        current_dist.append((dist, ia, ib))

    if len(current_dist) > 0:
        dist, ia, ib = sorted(current_dist, key=lambda k: k[0])[0]
        s_clusters.append((ia, ib))
"""

for n, (ia, centroid_a) in enumerate(d_clusters[1:], 1):
    current_clusters = d_clusters[n + 1:]
    current_dist = []

    for ib, centroid_b in current_clusters:
        dist = np.linalg.norm(centroid_a[1] - centroid_b[1])
        current_dist.append((dist, ia, ib))

    if len(current_dist) > 0:
        dist, ia, ib = sorted(current_dist, key=lambda k: k[0])[0]
        s_clusters.append((ia, ib))

s_clusters


[(1, 5), (2, 3), (3, 5), (4, 7), (5, 7), (6, 7)]

In [8]:
i_clusters = [i for i in range(len(all_clusters))]

for i in i_clusters:
    aux_clusters = all_clusters[i + 1:]




In [9]:
plotter = pv.Plotter()

for n, (a, b) in enumerate(s_clusters):
    c_a = sorted_data[a]
    c_b = sorted_data[b]

    s_data = [*c_a] + [*c_b]
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid_a = np.mean(list(c_a), axis=0)
    centroid_b = np.mean(list(c_b), axis=0)

    plotter.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=5
    )

    plotter.add_points(
        centroid_a,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=15
    )

    plotter.add_points(
        centroid_b,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=15
    )


plotter.add_axes()
plotter.show_grid()
plotter.show()

Widget(value='<iframe src="http://localhost:44965/index.html?ui=P_0x7f01b7178cd0_1&reconnect=auto" class="pyvi…

In [10]:
plotter = pv.Plotter()

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plotter.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=5
    )

    plotter.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=15
    )


plotter.add_axes()
plotter.show_grid()
plotter.show()

Widget(value='<iframe src="http://localhost:44965/index.html?ui=P_0x7f01b717a850_2&reconnect=auto" class="pyvi…